# F4·F5 EarnMore 시험 (notes/funnel_protocol.md)
런타임: **GPU**. 위에서부터 차례로 실행합니다. 결과는 Drive `data/funnel/earnmore/`에 저장됩니다.

- 공식 코드 DVampire/EarnMore @810ff59, 수정 없음.
- **성과 비노출:** 학습 2007-09-26~2014-01-01, 검증 2014-01-01~2015-01-01. 논문의 학습 구간 안이므로 시험 구간을 쓰지 않습니다. 출력은 파일로만 보내고 화면에는 설치 결과와 시간만 냅니다.
- 2 에피소드만 돌려 시간을 잽니다. 이 노트북으로 재현 판정을 하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, time, json
OUT = pathlib.Path('/content/drive/MyDrive/data/funnel/earnmore'); OUT.mkdir(parents=True, exist_ok=True)
START = time.time(); (OUT/'log.txt').open('a').write(f'start {time.ctime()}\n')
!nvidia-smi -L

In [ ]:
%%bash
set -e
cd /content
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
./bin/micromamba create -y -q -p /content/em -c conda-forge python=3.9 pip 'setuptools<66' 'wheel<0.40'
[ -d EarnMore ] || git clone -q https://github.com/DVampire/EarnMore.git
cd EarnMore && git checkout -q 810ff59 && git log -1 --format='%h %ad' --date=short

In [ ]:
%%bash
cd /content
P=/content/em/bin/pip
$P install -q torch==1.13.0+cu117 --extra-index-url https://download.pytorch.org/whl/cu117 2>&1 | tail -2
[ -d qlib ] || git clone -q https://github.com/microsoft/qlib.git
(cd qlib && $P install -q . 2>&1 | tail -2)
cd EarnMore && $P install -q -r requirements.txt 2>&1 | tail -3
/content/em/bin/python -c "import torch,gym,pandas,mmengine;print(torch.__version__,torch.cuda.is_available(),gym.__version__,pandas.__version__)"

In [ ]:
%%bash
cd /content/EarnMore
mkdir -p /content/out
export PYTHONPATH=$PWD
/content/em/bin/python tools/preprocess.py > /content/out/preprocess.log 2>&1; echo preprocess exit $?
ls datasets/dj30/features | wc -l
/content/em/bin/python tools/make_scripts.py --config configs/mask_sac_portfolio_management.py --mask --action_wrapper_method softmax --num_episodes 2000 --dataset dj30 --num_stocks 28 --buffer_size 10000 --repeat_times 128 --gpu_id 0 --days 10 --lr 1e-5 --act_lr 1e-5 --cri_lr 1e-5 --rep_lr 1e-5 --beta_lr 1e-5 --seed 42 --T 0.1 --data_group 0 > /content/out/make_scripts.log 2>&1; echo make_scripts exit $?
git status --short | grep -v features | head

위 칸의 마지막 줄에 나온 생성 파일을 다음 칸이 자동으로 찾습니다. 오류가 나면 그대로 알려 주세요.

In [ ]:
import glob, subprocess, shutil, re
os.chdir('/content/EarnMore')
new = subprocess.check_output('git status --short --untracked-files=all', shell=True, text=True).split('\n')
cfgs = [l[3:] for l in new if l[3:].endswith('.py') and 'configs' in l]
print(cfgs[:5])
cfg = cfgs[0]
env = dict(os.environ, PYTHONPATH=os.getcwd())
opts = ['num_episodes=2','train_start_date=2007-09-26','val_start_date=2014-01-01','test_start_date=2015-01-01','test_end_date=2015-06-01']
t0 = time.time()
with open('/content/out/smoke.log','w') as f:
    rc = subprocess.call(['/content/em/bin/python','tools/train.py','--config',cfg,'--workdir','/content/out/work','--cfg-options',*opts], stdout=f, stderr=f, env=env)
sec = time.time()-t0
txt = open('/content/out/smoke.log', errors='ignore').read()
eps = len(re.findall(r'Train Episode: \[', txt))
r = {'rc':rc,'seconds':sec,'episodes':eps,'hours_per_seed_at_2000':sec/max(eps,1)*2000/3600,'gpu':subprocess.getoutput('nvidia-smi -L'),'config':cfg}
print(json.dumps(r, indent=1))
(OUT/'f4_timing.json').write_text(json.dumps(r, indent=1))

In [ ]:
# 오류가 나면 마지막 줄만 보여 줍니다(성과 표는 출력하지 않음).
import subprocess
for n in ['preprocess.log','make_scripts.log','smoke.log']:
    shutil.copy('/content/out/'+n, OUT/n)
print(subprocess.getoutput("grep -nE 'Error|Traceback|error:' /content/out/smoke.log | tail -5"))
print('작업 시간(시간):', round((time.time()-START)/3600,2))
(OUT/'log.txt').open('a').write(f'end {time.ctime()}\n')